In [40]:
import os

# =========================================================
# USER CONFIGURATION — CHANGE ONLY VALUES HERE
# =========================================================

# OpenRouter
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")

# Model settings
MODEL_ID = "claude-sonnet-4"
MODEL_TEMPERATURE = 0.2

# Optimization limits
MIN_PASS_IMAGES_REQUIRED = 3        # safety guard
OPTIMIZATION_ITERATIONS = 5


# Request behavior
OPENROUTER_TIMEOUT_SECONDS = 120
OPENROUTER_RETRIES = 3

# Parallel processing
# Number of concurrent API calls (higher = faster but may hit rate limits)
# Recommended values:
#   - 3-5: Safe, works with most APIs
#   - 5-10: Faster, good for OpenRouter/Claude
#   - 10+: Very fast, but may hit rate limits (check your API provider's limits)
MAX_WORKERS = 10

# Data / storage
PROJECTS_DIR_PATH = "./projects"

# Base rules (fundamental rules that must always be preserved in all prompts)
BASE_RULES = [
    "It is enough that one of the Bird vehicles in the picture is PASS to grant a PASS for the image."
]

# =========================================================


In [41]:
# ========================================================= 
# DEFAULT PROMPTS (USED FOR NEW PROJECTS) # 
# =========================================================

# system-prompt: system-mobile-v1.3
# Base rules are included in the system prompt to ensure they're always preserved
BASE_RULES_TEXT = "\n\n".join([f"• {rule}" for rule in BASE_RULES]) if BASE_RULES else ""

DEFAULT_SYSTEM_PROMPT = f"""
You are a micromobility parking enforcement officer.
Analyze this parking photo of a shared e-scooter or bicycle.
Rate it as PASS or FAIL and provide feedback accordingly.

Condense your feedback into a single, short sentence.
If FAIL, make it actionable so the rider knows what they need to improve.

BASE RULES (must always be followed):
{BASE_RULES_TEXT}
""".strip()


# base-user-prompt: mobile-parking-review-v1.1
DEFAULT_BASE_USER_PROMPT = """
Apply the following rules:

1. The vehicle must be fully visible in the photo. It cannot be cut off. The photo should include enough surrounding space to judge whether the vehicle is parked appropriately.
2. Vehicles must not block sidewalks or pedestrian paths.
3. Parking on sidewalks is only ok if the vehicle is fully to the side and does not obstruct pedestrian flow.
4. Parking in the furniture zone of the sidewalk/street is ok and better than if parked in the middle of the sidewalk
5. Parking in designated micromobility spaces is preferred, but optional. Designated parking spaces are usually marked with paint on the ground or clear sign posts or other signage.
6. Vehicles must not block streets, driveways, public transport stations or entrances
7. Parking near crosswalks, ramps, or handicap-accessible areas is not allowed.
8. The vehicle must be upright and not tipped over.

Only refer to the brand of the scooter or bike if you are absolutely sure you detected the brand correctly (Bird, Spin).
""".strip()


In [42]:
import os
import json
import base64
import mimetypes
import time
import logging
import threading
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import gradio as gr
import pandas as pd
from PIL import Image

# =========================================================
# LOGGING SETUP
# =========================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# =========================================================
# USER CONFIG — CHANGE ONLY HERE
# =========================================================
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
MODEL_ID = "anthropic/claude-sonnet-4"
MODEL_TEMPERATURE = 0.2

OPTIMIZATION_ITERATIONS = 5          # hard-coded number of prompt rewrites to try
MIN_PASS_IMAGES_REQUIRED = 3         # safety guard so recall isn't meaningless

OPENROUTER_TIMEOUT_SECONDS = 120
OPENROUTER_RETRIES = 3

# Global stop event for cancellation
stop_event = threading.Event()
# =========================================================


# -------------------- HTTP helper (retries) --------------------

def post_with_retries(
    url: str,
    headers: dict,
    payload: dict,
    timeout_s: int = OPENROUTER_TIMEOUT_SECONDS,
    retries: int = OPENROUTER_RETRIES,
) -> dict:
    """
    POST with retries + exponential backoff.
    Returns response.json() on success.
    Raises RuntimeError on final failure.
    """
    logger.info(f"Making request to {url} (timeout={timeout_s}s, retries={retries})")
    last_err: Optional[str] = None
    for attempt in range(1, retries + 1):
        try:
            logger.debug(f"Attempt {attempt}/{retries}")
            r = requests.post(url, headers=headers, json=payload, timeout=timeout_s)
            if r.status_code == 200:
                data = r.json()
                # Check for provider errors in the response (status 200 but error in body)
                if "error" in data:
                    error_info = data.get("error", {})
                    error_msg = error_info.get("message", "Unknown error")
                    # Try to extract more details from metadata
                    if "metadata" in error_info and "raw" in error_info["metadata"]:
                        try:
                            import json as json_module
                            raw_error = json_module.loads(error_info["metadata"]["raw"])
                            if "error" in raw_error and "message" in raw_error["error"]:
                                error_msg = raw_error["error"]["message"]
                        except:
                            pass
                    last_err = f"Provider error: {error_msg}"
                    logger.warning(f"Request returned error in response: {last_err}")
                else:
                    logger.info(f"Request successful on attempt {attempt}")
                    return data
            else:
                last_err = f"HTTP {r.status_code}: {r.text[:600]}"
                logger.warning(f"Request failed with status {r.status_code}: {last_err}")
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout, BrokenPipeError) as e:
            last_err = f"Connection error: {type(e).__name__}: {str(e)}"
            logger.warning(f"Request connection error on attempt {attempt}: {e}")
        except Exception as e:
            last_err = str(e)
            logger.warning(f"Request exception on attempt {attempt}: {type(e).__name__}: {e}")

        if attempt < retries:
            wait_time = 2 ** (attempt - 1)
            logger.info(f"Waiting {wait_time}s before retry...")
            time.sleep(wait_time)

    error_msg = f"OpenRouter request failed after {retries} attempts. Last error: {last_err}"
    logger.error(error_msg)
    raise RuntimeError(error_msg)


# -------------------- Label + file helpers --------------------

def infer_label(filename: str) -> str:
    name = os.path.basename(filename).upper()
    if name.startswith("PASS"):
        return "PASS"
    if name.startswith("FAIL"):
        return "FAIL"
    raise ValueError(f"Filename must start with PASS or FAIL: {os.path.basename(filename)}")

def to_data_url(filepath: str, max_size_mb: float = 4.5) -> str:
    """
    Convert image file to data URL, compressing if necessary.
    
    Args:
        filepath: Path to image file
        max_size_mb: Maximum size in MB for base64-encoded image (default 4.5 MB to stay under 5 MB limit)
    
    Returns:
        Data URL string
    """
    mime, _ = mimetypes.guess_type(filepath)
    if not mime or not mime.startswith("image/"):
        mime = "image/jpeg"  # Default to JPEG
    
    # Read original file
    with open(filepath, "rb") as f:
        original_data = f.read()
    
    # Check if we need to compress (5 MB = 5242880 bytes, but base64 adds ~33% overhead)
    # So we want to keep base64 size under ~4.5 MB to be safe
    max_size_bytes = int(max_size_mb * 1024 * 1024)
    
    # Estimate base64 size (base64 is ~33% larger than binary)
    estimated_base64_size = len(original_data) * 4 // 3
    
    if estimated_base64_size <= max_size_bytes:
        # No compression needed
        b64 = base64.b64encode(original_data).decode("utf-8")
        logger.debug(f"Image {os.path.basename(filepath)}: {len(original_data)} bytes, no compression needed")
    else:
        # Need to compress
        logger.info(f"Compressing image {os.path.basename(filepath)}: {len(original_data)} bytes -> target < {max_size_bytes} bytes")
        try:
            # Open and compress image
            img = Image.open(BytesIO(original_data))
            
            # Convert RGBA to RGB if necessary (removes alpha channel)
            if img.mode in ("RGBA", "LA", "P"):
                # Create white background
                background = Image.new("RGB", img.size, (255, 255, 255))
                if img.mode == "P":
                    img = img.convert("RGBA")
                background.paste(img, mask=img.split()[-1] if img.mode in ("RGBA", "LA") else None)
                img = background
            elif img.mode != "RGB":
                img = img.convert("RGB")
            
            # Calculate compression quality and size
            # Start with quality 85 and reduce if needed
            quality = 85
            max_dimension = 2048  # Maximum width or height
            
            # Resize if image is too large
            if max(img.size) > max_dimension:
                ratio = max_dimension / max(img.size)
                new_size = (int(img.size[0] * ratio), int(img.size[1] * ratio))
                img = img.resize(new_size, Image.Resampling.LANCZOS)
                logger.debug(f"Resized image to {new_size}")
            
            # Try to compress to target size
            for attempt in range(5):  # Try up to 5 compression attempts
                output = BytesIO()
                img.save(output, format="JPEG", quality=quality, optimize=True)
                compressed_data = output.getvalue()
                estimated_base64_size = len(compressed_data) * 4 // 3
                
                if estimated_base64_size <= max_size_bytes:
                    b64 = base64.b64encode(compressed_data).decode("utf-8")
                    logger.info(f"Compressed image {os.path.basename(filepath)}: {len(original_data)} -> {len(compressed_data)} bytes (quality={quality})")
                    break
                else:
                    # Reduce quality and try again
                    quality = max(30, quality - 15)  # Don't go below quality 30
                    if attempt == 4:
                        # Last attempt - use what we have
                        b64 = base64.b64encode(compressed_data).decode("utf-8")
                        logger.warning(f"Image {os.path.basename(filepath)} still large after compression: {len(compressed_data)} bytes (quality={quality})")
            else:
                # Fallback: use original if compression failed
                b64 = base64.b64encode(original_data).decode("utf-8")
                logger.warning(f"Compression failed for {os.path.basename(filepath)}, using original")
                
        except Exception as e:
            # If compression fails, try original
            logger.warning(f"Error compressing image {os.path.basename(filepath)}: {e}, using original")
            b64 = base64.b64encode(original_data).decode("utf-8")
    
    return f"data:{mime};base64,{b64}"

def parse_json_strict(text: str) -> Dict[str, Any]:
    text = (text or "").strip()
    if text.startswith("{") and text.endswith("}"):
        return json.loads(text)

    # salvage first {...}
    s = text.find("{")
    e = text.rfind("}")
    if s != -1 and e != -1 and e > s:
        return json.loads(text[s:e+1])

    raise ValueError("Model response is not valid JSON")

def compute_metrics(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    # PASS is positive class
    tp = sum(1 for r in rows if r["gt"] == "PASS" and r["pred"] == "PASS")
    fn = sum(1 for r in rows if r["gt"] == "PASS" and r["pred"] == "FAIL")
    fp = sum(1 for r in rows if r["gt"] == "FAIL" and r["pred"] == "PASS")
    tn = sum(1 for r in rows if r["gt"] == "FAIL" and r["pred"] == "FAIL")

    recall = (tp / (tp + fn) * 100.0) if (tp + fn) else 0.0
    precision = (tp / (tp + fp) * 100.0) if (tp + fp) else 0.0

    return {
        "tp": tp, "fn": fn, "fp": fp, "tn": tn,
        "recall_pass_pct": round(recall, 2),
        "precision_pass_pct": round(precision, 2),
    }


# -------------------- OpenRouter calls --------------------

def openrouter_classify(system_prompt: str, user_prompt: str, image_data_url: str) -> Dict[str, Any]:
    logger.debug("Starting openrouter_classify")
    if not OPENROUTER_API_KEY:
        error_msg = "Missing OPENROUTER_API_KEY environment variable"
        logger.error(error_msg)
        raise RuntimeError(error_msg)

    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:7860",
        "X-Title": "Gradio Prompt Optimizer (No Projects)",
    }

    json_contract = """
Return ONLY valid JSON (no markdown, no extra text).
Schema:
{
  "decision": "PASS" | "FAIL",
  "reason": "short reason (max 25 words)"
}
""".strip()

    payload = {
        "model": MODEL_ID,
        "temperature": MODEL_TEMPERATURE,
        "messages": [
            {"role": "system", "content": (system_prompt or "").strip()},
            {"role": "user", "content": [
                {"type": "text", "text": (user_prompt or "").strip() + "\n\n" + json_contract},
                {"type": "image_url", "image_url": {"url": image_data_url}},
            ]},
        ],
    }

    try:
        logger.debug(f"Calling OpenRouter API with model {MODEL_ID}")
        data = post_with_retries(url, headers, payload)
        logger.debug("Received response from OpenRouter")
        
        # Check for errors in response
        if "error" in data:
            error_info = data.get("error", {})
            error_msg = error_info.get("message", "Unknown error")
            # Try to extract more details from metadata
            if "metadata" in error_info and "raw" in error_info["metadata"]:
                try:
                    raw_error = json.loads(error_info["metadata"]["raw"])
                    if "error" in raw_error and "message" in raw_error["error"]:
                        error_msg = raw_error["error"]["message"]
                except:
                    pass
            raise RuntimeError(f"API returned error: {error_msg}")
        
        if "choices" not in data or len(data["choices"]) == 0:
            error_msg = f"Invalid response structure: {data}"
            logger.error(error_msg)
            raise ValueError(error_msg)
        
        content = data["choices"][0]["message"]["content"]
        logger.debug(f"Response content length: {len(content)}")
        
        parsed = parse_json_strict(content)
        logger.debug(f"Parsed JSON successfully: {parsed}")

        decision = str(parsed.get("decision", "")).upper().strip()
        if decision not in ("PASS", "FAIL"):
            error_msg = f"Invalid decision: {decision}"
            logger.error(error_msg)
            raise ValueError(error_msg)

        result = {"decision": decision, "reason": parsed.get("reason", "")}
        logger.debug(f"Classification result: {result}")
        return result
    except Exception as e:
        logger.error(f"Error in openrouter_classify: {type(e).__name__}: {str(e)}")
        raise

def propose_next_user_prompt(
    system_prompt: str,
    current_user_prompt: str,
    false_negatives: List[Dict[str, Any]],
    false_positives: List[Dict[str, Any]],
) -> str:
    logger.info(f"Proposing next user prompt (FN: {len(false_negatives)}, FP: {len(false_positives)})")
    if not OPENROUTER_API_KEY:
        error_msg = "Missing OPENROUTER_API_KEY environment variable"
        logger.error(error_msg)
        raise RuntimeError(error_msg)

    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:7860",
        "X-Title": "Gradio Prompt Optimizer (No Projects)",
    }

    fn_lines = "\n".join([
        f"- {x['file']}: predicted FAIL but should PASS. Model reason: {x.get('reason','')}"
        for x in false_negatives[:12]
    ]) or "- None"

    fp_lines = "\n".join([
        f"- {x['file']}: predicted PASS but should FAIL. Model reason: {x.get('reason','')}"
        for x in false_positives[:12]
    ]) or "- None"

    # Get base rules text for the optimizer
    base_rules_text = ""
    try:
        if BASE_RULES:
            base_rules_text = "\n\nBASE RULES (must be preserved in all prompts - these are in the system prompt and should be reflected in user prompt logic):\n" + "\n".join([f"- {rule}" for rule in BASE_RULES])
    except NameError:
        pass  # BASE_RULES not defined, skip
    
    prompt = f"""
You are a prompt engineer.

Goal (strict priority):
1) Maximize PASS recall (catch as many true PASS as possible)
2) Then maximize PASS precision (avoid false PASS)

You MUST output ONLY the revised USER PROMPT text (no commentary, no markdown).

Constraints:
- Do not change the system prompt.
- Keep it clear and self-contained.
- Ensure the classifier returns JSON only with decision PASS/FAIL and short reason.
- The system prompt contains BASE RULES that must always be followed - ensure your revised user prompt aligns with these rules.
{base_rules_text}

Current USER PROMPT:
{current_user_prompt}

Edge cases - False Negatives (GT PASS, predicted FAIL):
{fn_lines}

Edge cases - False Positives (GT FAIL, predicted PASS):
{fp_lines}

Rewrite the USER PROMPT to reduce these errors. Output ONLY the new USER PROMPT text.
""".strip()

    payload = {
        "model": MODEL_ID,
        "temperature": MODEL_TEMPERATURE,
        "messages": [
            {"role": "system", "content": (system_prompt or "").strip()},
            {"role": "user", "content": prompt},
        ],
    }

    try:
        logger.debug("Calling optimizer LLM")
        data = post_with_retries(url, headers, payload)
        
        if "choices" not in data or len(data["choices"]) == 0:
            error_msg = f"Invalid response structure from optimizer: {data}"
            logger.error(error_msg)
            raise ValueError(error_msg)
        
        new_prompt = (data["choices"][0]["message"]["content"] or "").strip()
        if not new_prompt:
            error_msg = "Empty prompt returned from optimizer step"
            logger.error(error_msg)
            raise RuntimeError(error_msg)
        
        logger.info(f"New prompt generated (length: {len(new_prompt)} chars)")
        return new_prompt
    except Exception as e:
        logger.error(f"Error in propose_next_user_prompt: {type(e).__name__}: {str(e)}")
        raise


# -------------------- Evaluation + optimization loop --------------------

@dataclass
class ImageItem:
    file: str
    gt: str
    data_url: str

def evaluate_single_item(
    item: ImageItem,
    system_prompt: str,
    user_prompt: str,
    idx: int,
    total: int
) -> Dict[str, Any]:
    """Evaluate a single image item (used for parallel processing)"""
    # Check for stop request
    if stop_event.is_set():
        raise InterruptedError("Optimization stopped by user")
    
    try:
        logger.debug(f"Evaluating item {idx}/{total}: {os.path.basename(item.file)}")
        out = openrouter_classify(system_prompt, user_prompt, item.data_url)
        pred = out["decision"]
        reason = out.get("reason", "")
        logger.debug(f"Item {idx} result: {pred}")
        return {
            "file": os.path.basename(item.file),
            "gt": item.gt,
            "pred": pred,
            "reason": reason,
            "error": False
        }
    except InterruptedError:
        raise  # Re-raise stop requests
    except Exception as e:
        logger.warning(f"Error evaluating item {idx} ({os.path.basename(item.file)}): {type(e).__name__}: {str(e)}")
        return {
            "file": os.path.basename(item.file),
            "gt": item.gt,
            "pred": "FAIL",  # conservative fallback
            "reason": f"ERROR: {str(e)[:180]}",
            "error": True
        }

def evaluate_prompt(
    system_prompt: str,
    user_prompt: str,
    items: List[ImageItem],
    progress: Optional[gr.Progress] = None,
    prefix: str = "Eval"
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    logger.info(f"Evaluating prompt on {len(items)} items (prefix: {prefix}) using {MAX_WORKERS} workers")
    rows: List[Dict[str, Any]] = []
    errors = 0
    total = len(items)
    
    # Use parallel processing for faster evaluation
    completed = 0
    rows_dict = {}  # Use dict to maintain order
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks
        future_to_idx = {
            executor.submit(evaluate_single_item, it, system_prompt, user_prompt, idx, total): idx
            for idx, it in enumerate(items, start=1)
        }
        
        # Process completed tasks as they finish
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            completed += 1
            
            # Check for stop request
            if stop_event.is_set():
                logger.warning(f"Stop requested during evaluation at item {completed}/{total}")
                # Cancel remaining tasks
                for f in future_to_idx:
                    f.cancel()
                raise InterruptedError("Optimization stopped by user")
            
            # Update progress
            if progress is not None:
                progress(completed / max(total, 1), desc=f"{prefix}: {completed}/{total}")
            
            try:
                result = future.result(timeout=OPENROUTER_TIMEOUT_SECONDS + 10)  # Add small buffer
                rows_dict[idx] = result
                if result.get("error", False):
                    errors += 1
            except InterruptedError:
                raise
            except TimeoutError as e:
                logger.error(f"Timeout processing item {idx}: {e}")
                rows_dict[idx] = {
                    "file": os.path.basename(items[idx-1].file),
                    "gt": items[idx-1].gt,
                    "pred": "FAIL",
                    "reason": f"ERROR: Timeout after {OPENROUTER_TIMEOUT_SECONDS}s",
                    "error": True
                }
                errors += 1
            except Exception as e:
                logger.error(f"Unexpected error processing item {idx}: {type(e).__name__}: {e}", exc_info=True)
                rows_dict[idx] = {
                    "file": os.path.basename(items[idx-1].file),
                    "gt": items[idx-1].gt,
                    "pred": "FAIL",
                    "reason": f"ERROR: {str(e)[:180]}",
                    "error": True
                }
                errors += 1
    
    # Reconstruct rows in original order
    # Ensure we have results for all items (fill missing ones)
    rows = []
    for idx in range(1, total + 1):
        if idx in rows_dict:
            row = rows_dict[idx].copy()
            row.pop("error", None)  # Remove error flag before returning
            rows.append(row)
        else:
            # Missing result - should not happen, but handle gracefully
            logger.warning(f"Missing result for item {idx}, creating fallback")
            rows.append({
                "file": os.path.basename(items[idx-1].file),
                "gt": items[idx-1].gt,
                "pred": "FAIL",
                "reason": "ERROR: Missing result",
            })
            errors += 1
    
    if progress is not None:
        progress(1.0, desc=f"{prefix}: done")

    metrics = compute_metrics(rows)
    metrics["invalid_json_or_error_count"] = errors
    logger.info(f"Evaluation complete: {metrics} (errors: {errors})")
    return rows, metrics

def better(a: Dict[str, Any], b: Dict[str, Any]) -> Dict[str, Any]:
    ma = a["metrics"]
    mb = b["metrics"]
    key_a = (ma["recall_pass_pct"], ma["precision_pass_pct"])
    key_b = (mb["recall_pass_pct"], mb["precision_pass_pct"])
    return a if key_a >= key_b else b

def run_optimizer(
    files: List[str],
    progress: gr.Progress = gr.Progress(track_tqdm=False),
):
    # Reset stop event at start
    stop_event.clear()
    
    logger.info("=" * 80)
    logger.info("Starting optimization run")
    logger.info("=" * 80)
    
    try:
        if not files:
            error_msg = "Upload images first (filenames must start with PASS_ or FAIL_)."
            logger.error(error_msg)
            raise ValueError(error_msg)

        logger.info(f"Received {len(files)} files")

        # Build items (data_urls created once)
        progress(0.0, desc="Preparing dataset...")
        logger.info("Preparing dataset...")
        items: List[ImageItem] = []
        pass_n = 0
        fail_n = 0
        bad = 0

        for f in files:
            try:
                gt = infer_label(f)
                if gt == "PASS":
                    pass_n += 1
                else:
                    fail_n += 1
                items.append(ImageItem(file=f, gt=gt, data_url=to_data_url(f)))
                logger.debug(f"Added item: {os.path.basename(f)} -> {gt}")
            except Exception as e:
                bad += 1
                logger.warning(f"Skipping file {os.path.basename(f)}: {e}")
                continue

        logger.info(f"Dataset prepared: total={len(items)}, PASS={pass_n}, FAIL={fail_n}, bad={bad}")

        if not items:
            error_msg = "No valid images found. Make sure filenames start with PASS_ or FAIL_."
            logger.error(error_msg)
            raise ValueError(error_msg)
        if pass_n < MIN_PASS_IMAGES_REQUIRED:
            error_msg = (
                f"Not enough PASS images to evaluate recall reliably "
                f"(found {pass_n}, need at least {MIN_PASS_IMAGES_REQUIRED})."
            )
            logger.error(error_msg)
            raise ValueError(error_msg)

        # Use DEFAULT prompts if available, otherwise use fallback
        try:
            system_prompt = DEFAULT_SYSTEM_PROMPT
            base_user_prompt = DEFAULT_BASE_USER_PROMPT
            logger.info("Using DEFAULT_SYSTEM_PROMPT and DEFAULT_BASE_USER_PROMPT")
        except NameError:
            # Fallback prompts if DEFAULT_* are not defined
            logger.warning("DEFAULT_SYSTEM_PROMPT or DEFAULT_BASE_USER_PROMPT not found, using fallback prompts")
            system_prompt = """You are a micromobility parking enforcement officer.
Analyze this parking photo of a shared e-scooter or bicycle.
Rate it as PASS or FAIL and provide feedback accordingly.

Condense your feedback into a single, short sentence.
If FAIL, make it actionable so the rider knows what they need to improve.""".strip()
            
            base_user_prompt = """Apply the following rules:

1. The vehicle must be fully visible in the photo. It cannot be cut off. The photo should include enough surrounding space to judge whether the vehicle is parked appropriately.
2. Vehicles must not block sidewalks or pedestrian paths.
3. Parking on sidewalks is only ok if the vehicle is fully to the side and does not obstruct pedestrian flow.
4. Parking in the furniture zone of the sidewalk/street is ok and better than if parked in the middle of the sidewalk
5. Parking in designated micromobility spaces is preferred, but optional. Designated parking spaces are usually marked with paint on the ground or clear sign posts or other signage.
6. Vehicles must not block streets, driveways, public transport stations or entrances
7. Parking near crosswalks, ramps, or handicap-accessible areas is not allowed.
8. The vehicle must be upright and not tipped over.

Only refer to the brand of the scooter or bike if you are absolutely sure you detected the brand correctly (Bird, Spin).""".strip()

        logger.info(f"System prompt length: {len(system_prompt)} chars")
        logger.info(f"Base user prompt length: {len(base_user_prompt)} chars")

        history: List[Dict[str, Any]] = []

        # Iteration 0 (base prompt)
        logger.info("Starting iteration 0 (base prompt evaluation)")
        progress(0.0, desc="Iteration 0: evaluating base prompt...")
        try:
            rows0, m0 = evaluate_prompt(system_prompt, base_user_prompt, items, progress=progress, prefix="Iter 0")
            history.append({"iteration": 0, "user_prompt": base_user_prompt, "metrics": m0, "rows": rows0})
            best = history[0]
            current_prompt = base_user_prompt
            logger.info(f"Iteration 0 complete: {m0}")
        except Exception as e:
            logger.error(f"Error in iteration 0: {type(e).__name__}: {str(e)}", exc_info=True)
            raise

        # Iterations 1..OPTIMIZATION_ITERATIONS
        for i in range(1, OPTIMIZATION_ITERATIONS + 1):
            # Check for stop request before starting iteration
            if stop_event.is_set():
                logger.warning(f"Stop requested before iteration {i}")
                raise InterruptedError("Optimization stopped by user")
            
            logger.info(f"Starting iteration {i}")
            try:
                progress(0.0, desc=f"Iteration {i}: building edge cases...")
                last_rows = history[-1]["rows"]
                false_neg = [r for r in last_rows if r["gt"] == "PASS" and r["pred"] == "FAIL"]
                false_pos = [r for r in last_rows if r["gt"] == "FAIL" and r["pred"] == "PASS"]
                logger.info(f"Iteration {i}: Found {len(false_neg)} false negatives, {len(false_pos)} false positives")

                # Check for stop request
                if stop_event.is_set():
                    logger.warning(f"Stop requested during iteration {i} (building edge cases)")
                    raise InterruptedError("Optimization stopped by user")

                progress(0.0, desc=f"Iteration {i}: proposing new prompt...")
                new_prompt = propose_next_user_prompt(
                    system_prompt=system_prompt,
                    current_user_prompt=current_prompt,
                    false_negatives=false_neg,
                    false_positives=false_pos,
                )

                # Check for stop request
                if stop_event.is_set():
                    logger.warning(f"Stop requested during iteration {i} (after prompt proposal)")
                    raise InterruptedError("Optimization stopped by user")

                progress(0.0, desc=f"Iteration {i}: evaluating new prompt...")
                rows_i, m_i = evaluate_prompt(system_prompt, new_prompt, items, progress=progress, prefix=f"Iter {i}")
                candidate = {"iteration": i, "user_prompt": new_prompt, "metrics": m_i, "rows": rows_i}
                history.append(candidate)
                logger.info(f"Iteration {i} complete: {m_i}")

                best = better(best, candidate)
                current_prompt = new_prompt
            except InterruptedError:
                # Re-raise interruption errors
                raise
            except Exception as e:
                logger.error(f"Error in iteration {i}: {type(e).__name__}: {str(e)}", exc_info=True)
                raise

        best_metrics = best["metrics"]
        best_prompt = best["user_prompt"]
        best_rows = best["rows"]

        logger.info(f"Optimization complete. Best iteration: {best['iteration']}")
        logger.info(f"Best metrics: {best_metrics}")

        fn = [r for r in best_rows if r["gt"] == "PASS" and r["pred"] == "FAIL"]
        fp = [r for r in best_rows if r["gt"] == "FAIL" and r["pred"] == "PASS"]

        df_history = pd.DataFrame([{
            "iteration": h["iteration"],
            "recall_pass_pct": h["metrics"]["recall_pass_pct"],
            "precision_pass_pct": h["metrics"]["precision_pass_pct"],
            "tp": h["metrics"]["tp"],
            "fn": h["metrics"]["fn"],
            "fp": h["metrics"]["fp"],
            "tn": h["metrics"]["tn"],
            "invalid_json_or_error_count": h["metrics"].get("invalid_json_or_error_count", 0),
        } for h in history])

        # Ensure DataFrames are properly formatted for Gradio
        df_fn = pd.DataFrame(fn) if fn else pd.DataFrame(columns=["file", "gt", "pred", "reason"])
        df_fp = pd.DataFrame(fp) if fp else pd.DataFrame(columns=["file", "gt", "pred", "reason"])
        
        # Replace NaN values with empty strings for Gradio compatibility
        df_history = df_history.fillna("")
        df_fn = df_fn.fillna("")
        df_fp = df_fp.fillna("")
        
        # Ensure all DataFrames have string types for all columns (Gradio requirement)
        for col in df_history.columns:
            df_history[col] = df_history[col].astype(str)
        for col in df_fn.columns:
            df_fn[col] = df_fn[col].astype(str)
        for col in df_fp.columns:
            df_fp[col] = df_fp[col].astype(str)

        summary = (
            f"Model: {MODEL_ID} | Temp: {MODEL_TEMPERATURE}\n"
            f"Dataset: total={len(items)} PASS={pass_n} FAIL={fail_n} bad={bad}\n"
            f"Best iteration: {best['iteration']}\n"
            f"Recall(PASS): {best_metrics['recall_pass_pct']}%\n"
            f"Precision(PASS): {best_metrics['precision_pass_pct']}%\n"
            f"TP:{best_metrics['tp']} FN:{best_metrics['fn']} FP:{best_metrics['fp']} TN:{best_metrics['tn']}\n"
            f"Invalid/Errors: {best_metrics.get('invalid_json_or_error_count', 0)}\n"
        )

        progress(1.0, desc="Done")
        logger.info("=" * 80)
        logger.info("Optimization run completed successfully")
        logger.info("=" * 80)
        
        # Ensure all return values are properly formatted
        logger.info(f"Preparing results: summary length={len(summary)}, prompt length={len(best_prompt)}")
        logger.info(f"DataFrames: history={len(df_history)} rows, fn={len(df_fn)} rows, fp={len(df_fp)} rows")
        
        # Validate DataFrames before returning
        try:
            # Convert to dict for Gradio if needed
            if len(df_history) > 0:
                logger.info(f"History DF columns: {list(df_history.columns)}")
            if len(df_fn) > 0:
                logger.info(f"FN DF columns: {list(df_fn.columns)}")
            if len(df_fp) > 0:
                logger.info(f"FP DF columns: {list(df_fp.columns)}")
            
            logger.info("Returning results to Gradio...")
            return summary, best_prompt, df_history, df_fn, df_fp
        except Exception as e:
            logger.error(f"Error preparing return values: {type(e).__name__}: {str(e)}", exc_info=True)
            # Return safe fallback values
            fallback_summary = f"Error preparing results: {str(e)}"
            fallback_df = pd.DataFrame()
            return fallback_summary, "", fallback_df, fallback_df, fallback_df
        
    except InterruptedError as e:
        logger.warning(f"Optimization interrupted: {str(e)}")
        # Return partial results if available
        if 'history' in locals() and len(history) > 0:
            best_metrics = best["metrics"]
            best_prompt = best["user_prompt"]
            best_rows = best["rows"]
            
            fn = [r for r in best_rows if r["gt"] == "PASS" and r["pred"] == "FAIL"]
            fp = [r for r in best_rows if r["gt"] == "FAIL" and r["pred"] == "PASS"]
            
            df_history = pd.DataFrame([{
                "iteration": h["iteration"],
                "recall_pass_pct": h["metrics"]["recall_pass_pct"],
                "precision_pass_pct": h["metrics"]["precision_pass_pct"],
                "tp": h["metrics"]["tp"],
                "fn": h["metrics"]["fn"],
                "fp": h["metrics"]["fp"],
                "tn": h["metrics"]["tn"],
                "invalid_json_or_error_count": h["metrics"].get("invalid_json_or_error_count", 0),
            } for h in history])
            
            df_fn = pd.DataFrame(fn) if fn else pd.DataFrame(columns=["file", "gt", "pred", "reason"])
            df_fp = pd.DataFrame(fp) if fp else pd.DataFrame(columns=["file", "gt", "pred", "reason"])
            
            # Replace NaN values with empty strings for Gradio compatibility
            df_history = df_history.fillna("")
            df_fn = df_fn.fillna("")
            df_fp = df_fp.fillna("")
            
            summary = (
                f"Model: {MODEL_ID} | Temp: {MODEL_TEMPERATURE}\n"
                f"Dataset: total={len(items)} PASS={pass_n} FAIL={fail_n} bad={bad}\n"
                f"Best iteration: {best['iteration']} (STOPPED EARLY)\n"
                f"Recall(PASS): {best_metrics['recall_pass_pct']}%\n"
                f"Precision(PASS): {best_metrics['precision_pass_pct']}%\n"
                f"TP:{best_metrics['tp']} FN:{best_metrics['fn']} FP:{best_metrics['fp']} TN:{best_metrics['tn']}\n"
                f"Invalid/Errors: {best_metrics.get('invalid_json_or_error_count', 0)}\n"
            )
            return summary, best_prompt, df_history, df_fn, df_fp
        else:
            # No results yet, return empty results
            summary = "Optimization stopped before completion. No results available."
            empty_df = pd.DataFrame()
            return summary, "", empty_df, empty_df, empty_df
    except Exception as e:
        logger.error(f"Fatal error in run_optimizer: {type(e).__name__}: {str(e)}", exc_info=True)
        raise


# -------------------- Gradio UI --------------------

def stop_optimization():
    """Stop the optimization process"""
    stop_event.set()
    logger.info("Stop button clicked - optimization will stop at next checkpoint")
    return "⚠️ Stop requested. Optimization will stop at next checkpoint..."

def run_optimizer_wrapper(files):
    """Wrapper to provide status updates"""
    if not files:
        empty_df = pd.DataFrame()
        return (
            "❌ Error: Please upload images first (filenames must start with PASS_ or FAIL_).",
            "", 
            empty_df, 
            empty_df, 
            empty_df,
            "❌ No files uploaded",
            0.0
        )
    
    try:
        # Update status at start
        status_msg = f"🔄 Running... Processing {len(files)} files"
        logger.info(status_msg)
        
        # Run the optimizer
        logger.info("Calling run_optimizer...")
        summary, best_prompt, df_history, df_fn, df_fp = run_optimizer(
            files,
            progress=gr.Progress(track_tqdm=False)
        )
        
        logger.info("run_optimizer returned successfully")
        logger.info(f"Received: summary type={type(summary)}, prompt type={type(best_prompt)}")
        logger.info(f"Received: df_history type={type(df_history)}, shape={df_history.shape if hasattr(df_history, 'shape') else 'N/A'}")
        logger.info(f"Received: df_fn type={type(df_fn)}, shape={df_fn.shape if hasattr(df_fn, 'shape') else 'N/A'}")
        logger.info(f"Received: df_fp type={type(df_fp)}, shape={df_fp.shape if hasattr(df_fp, 'shape') else 'N/A'}")
        
        # Success status
        status_msg = "✅ Optimization completed successfully!"
        logger.info(status_msg)
        logger.info("Returning results to Gradio UI...")
        
        return summary, best_prompt, df_history, df_fn, df_fp, status_msg, 1.0
        
    except InterruptedError as e:
        status_msg = f"⚠️ Optimization stopped by user: {str(e)}"
        logger.warning(status_msg)
        empty_df = pd.DataFrame()
        return (
            status_msg,
            "",
            empty_df,
            empty_df,
            empty_df,
            status_msg,
            0.0
        )
    except Exception as e:
        error_msg = f"❌ Error: {type(e).__name__}: {str(e)}"
        logger.error(error_msg, exc_info=True)
        empty_df = pd.DataFrame()
        return (
            error_msg,
            "",
            empty_df,
            empty_df,
            empty_df,
            error_msg,
            0.0
        )

with gr.Blocks(title="Prompt Optimizer (No Projects)") as demo:
    gr.Markdown("## Prompt Optimizer — PASS recall-first, then PASS precision (5 iterations)")
    
    with gr.Row():
        with gr.Column(scale=2):
            files = gr.File(
                file_count="multiple",
                type="filepath",
                label="Upload images (filenames must start with PASS_ or FAIL_)",
            )
        with gr.Column(scale=1):
            status_box = gr.Textbox(
                label="Status",
                value="⏸️ Ready - Upload images and click 'Run optimization'",
                interactive=False,
                lines=3
            )
            progress_bar = gr.Slider(
                minimum=0,
                maximum=1,
                value=0,
                step=0.01,
                label="Progress",
                visible=True,
                interactive=False
            )

    with gr.Row():
        run_btn = gr.Button("Run optimization", variant="primary")
        stop_btn = gr.Button("Stop", variant="stop")

    summary = gr.Textbox(lines=8, label="Summary (best recall/precision)")
    best_prompt = gr.Textbox(lines=12, label="Best optimized user prompt")
    history_df = gr.Dataframe(label="Iteration history (percentages + counts)")
    fn_df = gr.Dataframe(label="False negatives (GT PASS, Pred FAIL)")
    fp_df = gr.Dataframe(label="False positives (GT FAIL, Pred PASS)")

    run_btn.click(
        fn=run_optimizer_wrapper,
        inputs=[files],
        outputs=[summary, best_prompt, history_df, fn_df, fp_df, status_box, progress_bar],
    )
    
    stop_btn.click(
        fn=stop_optimization,
        inputs=[],
        outputs=[status_box],
    )


if __name__ == "__main__":
    demo.launch()


2025-12-15 16:15:36,963 - INFO - HTTP Request: GET http://127.0.0.1:7874/gradio_api/startup-events "HTTP/1.1 200 OK"
2025-12-15 16:15:36,967 - INFO - HTTP Request: HEAD http://127.0.0.1:7874/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


2025-12-15 16:15:37,876 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
